In [1]:
import numpy as np
from sklearn.datasets import fetch_openml
from tqdm import tqdm

In [4]:

NUM_INPUT = 784
NUM_OUTPUT = 50
DT = 0.005
T_MAX = 0.350

def test_snn():
    print("Loading test data, weights, and labels...")
    mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='liac-arff')
    
    # Use the last 10,000 images for the test set
    images = mnist.data[60000:] / 255.0
    true_labels = mnist.target[60000:].astype(int)
    
    weights = np.load('vdsp_weights.npy')
    assigned_labels = np.load('assigned_labels.npy')
    
    correct_predictions = 0
    v_pre = np.zeros(NUM_INPUT)
    v_post = np.zeros(NUM_OUTPUT)
    
    # Pre-allocate states for matching training dynamics
    n_adapt = np.zeros(NUM_OUTPUT)
    refractory_pre = np.zeros(NUM_INPUT)
    refractory_post = np.zeros(NUM_OUTPUT)
    wta_clamp = np.zeros(NUM_OUTPUT)

    for img_idx, img in tqdm(enumerate(images), total=len(images), desc="Testing SNN"):
        # Reset states for the new image
        v_pre.fill(0.0)
        v_post.fill(0.0)
        n_adapt.fill(0.0)
        refractory_pre.fill(0.0)
        refractory_post.fill(0.0)
        wta_clamp.fill(0.0)
        
        neuron_spikes = np.zeros(NUM_OUTPUT)
        
        for t in np.arange(0, T_MAX, DT):
            # --- Update Input Neurons (LIF) ---
            active_pre = refractory_pre <= 0
            v_pre[active_pre] += (DT / 0.03) * (-v_pre[active_pre] + img[active_pre] + 0.5)
            
            spikes_pre = v_pre >= 1.0
            v_pre[spikes_pre] = -1.0
            refractory_pre[spikes_pre] = 0.005
            refractory_pre -= DT
            
            # --- Update Output Neurons (ALIF) ---
            active_post = (refractory_post <= 0) & (wta_clamp <= 0)
            I_post = spikes_pre.astype(float) @ weights
            
            v_post[active_post] += (DT / 0.03) * (-v_post[active_post] + I_post[active_post] - n_adapt[active_post])
            n_adapt -= (DT / 1.0) * n_adapt
            
            spikes_post = v_post >= 1.0
            
            # --- Apply Winner-Take-All ---
            if spikes_post.any():
                winner_idx = np.argmax(v_post)
                neuron_spikes[winner_idx] += 1
                
                v_post[winner_idx] = 0.0
                refractory_post[winner_idx] = 0.005
                n_adapt[winner_idx] += 0.01
                
                # Clamp other competitive neurons
                wta_clamp[:] = 0.010
                wta_clamp[winner_idx] = 0.0
                v_post[wta_clamp > 0] = 0.0
                
            refractory_post -= DT
            wta_clamp -= DT
                                
        # Predict class based on the neuron with max spikes
        if neuron_spikes.sum() > 0:
            predicted_neuron = np.argmax(neuron_spikes)
            predicted_label = assigned_labels[predicted_neuron]
            
            if predicted_label == true_labels[img_idx]:
                correct_predictions += 1
            
    final_acc = (correct_predictions / len(images)) * 100
    print(f"Final Test Accuracy: {final_acc:.2f}%")

In [5]:
test_snn()

Loading test data, weights, and labels...


Testing SNN: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 10000/10000 [01:01<00:00, 161.34it/s]

Final Test Accuracy: 68.45%
